In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import torch
from utils import *
from Experiment import Experiment
from ReasoningGraph import ReasoningGraph
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
# model_name = "Qwen/Qwen3-4B-Thinking-2507"
# model_name = "Qwen/Qwen3-1.7B"
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
num_problems = 3
num_rollouts = 5
temperature = 1.0

In [4]:
dataset_name = 'openai/gsm8k'
problems = load_problems_from_dataset(dataset=dataset_name, num_problems=num_problems)

for i, p in enumerate(problems[:3]):
    print(f"\nProblem {i+1}: {p['question'][:100]}...")
    print(f"Ground truth: {p['ground_truth']}")

Loading 3 problems from openai/gsm8k:test

Problem 1: Linus works for a trading company. He buys a mobile device for $20 and sells it for twice the amount...
Ground truth: 120.0

Problem 2: James loves to go swimming and has to swim across a 20-mile lake.  He can swim at a pace of 2 miles ...
Ground truth: 17.0

Problem 3: Lindsay is doing the laundry, and thinks she has missed some socks. There are 50 socks that need was...
Ground truth: 15.0


In [5]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Some parameters are on the meta device because they were offloaded to the disk.


In [ ]:
# Create an experiment
experiment = Experiment()

# Setup new experiment
experiment.setup_new(
    model_name=model_name,
    dataset_name=dataset_name,
    problems=problems,
    num_rollouts=num_rollouts,
    temperature=temperature
)


Created experiment at: experiments/experiment_20251027_225038 with config:
{'model_name': 'Qwen/Qwen2.5-Math-1.5B-Instruct', 'dataset_name': 'gsm8k', 'num_problems': 1, 'num_rollouts': 2, 'temperature': 1.0, 'timestamp': '20251027_225038'}


In [16]:
# Conduct the experiment with initiailized model and tokenizer
experiment.conduct_experiment(model, tokenizer)


Problem 1/1



Experiment completed! Results saved to: experiments/experiment_20251027_225038


In [ ]:
# Unload model and release CPU/GPU memory
del model
gc.collect() 
if torch.cuda.is_available():
    torch.cuda.empty_cache()

: 

In [24]:
tokens = [tokenizer.decode(id) for id in output_ids[len(model_inputs.input_ids[0]):]]
entropies = reasoning_graph.metrics
visualize_tokens(tokens, entropies)

In [ ]:
# Example of loading a saved reasoning graph
loaded_graph = ReasoningGraph.load("/Users/adityashukzy/Documents/GitHub/MAT1510-Project/experiments/experiment_20251027_155017/problem_258/rollout_0")

# Verify the loaded data
print("Metrics length:", len(loaded_graph.metrics))
print("Probability distributions shape:", loaded_graph.prob_distributions[0].shape)
print("Node cutoff value:", loaded_graph.node_cutoff)

# You can now use this loaded graph for visualization or analysis

Metrics length: 185
Probability distributions shape: torch.Size([151936])
Node cutoff value: 0.0986328125


: 